In [18]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# ======================
# LOAD DATA
# ======================
df = pd.read_csv(r"E:\bankruptcy-risk-system\data\processed\final_features.csv")

# ======================
# SPLIT FEATURES & TARGET
# ======================
X = df.drop(columns=["company_name", "default_flag"])
y = df["default_flag"]
# ======================
# REMOVE INF VALUES
# ======================
X = X.replace([np.inf, -np.inf], np.nan)

# ======================
# HANDLE EXTREME VALUES (CLIPPING)
# ======================
X = X.clip(lower=-1e6, upper=1e6)

# ======================
# FILL MISSING VALUES
# ======================
X = X.fillna(X.median())
# ======================
# TRAIN TEST SPLIT
# ======================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ======================
# SCALING (for LR only)
# ======================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ======================
# MODELS
# ======================

# 1. Logistic Regression
lr = LogisticRegression(class_weight='balanced', max_iter=1000)

# 2. Random Forest
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    class_weight='balanced',
    random_state=42
)

# 3. XGBoost
xgb = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    scale_pos_weight=(len(y_train[y_train==0]) / len(y_train[y_train==1])),
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

# ======================
# TRAIN MODELS
# ======================
lr.fit(X_train_scaled, y_train)
rf.fit(X_train, y_train)
xgb.fit(X_train, y_train)

# ======================
# PREDICTIONS
# ======================
lr_pred = lr.predict(X_test_scaled)
rf_pred = rf.predict(X_test)
xgb_pred = xgb.predict(X_test)
print("Unique predictions:", np.unique(xgb_pred))

y_proba = xgb.predict_proba(X_test)[:, 1]

# try lower threshold
threshold = 0.3

y_pred_custom = (y_proba > threshold).astype(int)

from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred_custom))

# ======================
# EVALUATION FUNCTION
# ======================
def evaluate(name, y_true, y_pred):
    print(f"\n===== {name} =====")
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("ROC-AUC:", roc_auc_score(y_true, y_pred))
    print(classification_report(y_true, y_pred))

# ======================
# EVALUATE ALL MODELS
# ======================
evaluate("Logistic Regression", y_test, lr_pred)
evaluate("Random Forest", y_test, rf_pred)
evaluate("XGBoost", y_test, xgb_pred)

# ======================
# FEATURE IMPORTANCE (RF)
# ======================
importances = pd.Series(rf.feature_importances_, index=X.columns)
print("\nTop Features:\n", importances.sort_values(ascending=False).head(10))



Unique predictions: [0 1]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1001
           1       1.00      1.00      1.00        70

    accuracy                           1.00      1071
   macro avg       1.00      1.00      1.00      1071
weighted avg       1.00      1.00      1.00      1071


===== Logistic Regression =====
Accuracy: 0.8683473389355743
ROC-AUC: 0.8365634365634366
              precision    recall  f1-score   support

           0       0.98      0.87      0.93      1001
           1       0.31      0.80      0.44        70

    accuracy                           0.87      1071
   macro avg       0.65      0.84      0.68      1071
weighted avg       0.94      0.87      0.89      1071


===== Random Forest =====
Accuracy: 0.9990662931839402
ROC-AUC: 0.9928571428571429
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1001
           1       1.00      0.99    

e:\bankruptcy-risk-system\venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [00:05:05] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [19]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# ======================
# LOAD DATA
# ======================
df = pd.read_csv(r"E:\bankruptcy-risk-system\data\processed\final_features.csv")

# ======================
# SPLIT FEATURES & TARGET
# ======================
X = df.drop(columns=["company_name", "default_flag"])
y = df["default_flag"]

# ======================
# TRAIN TEST SPLIT
# ======================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ======================
# PREPROCESSING (NO LEAKAGE)
# ======================

# Train
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_train = X_train.clip(lower=-1e6, upper=1e6)

train_median = X_train.median()
X_train = X_train.fillna(train_median)

# Test (use SAME median)
X_test = X_test.replace([np.inf, -np.inf], np.nan)
X_test = X_test.clip(lower=-1e6, upper=1e6)
X_test = X_test.fillna(train_median)

# ======================
# SCALING (for LR only)
# ======================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ======================
# MODELS
# ======================

# Logistic Regression
lr = LogisticRegression(class_weight='balanced', max_iter=1000)

# Random Forest
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    class_weight='balanced',
    random_state=42
)

# XGBoost
scale_pos_weight = len(y_train[y_train == 0]) / len(y_train[y_train == 1])

xgb = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=scale_pos_weight * 3,  # 🔥 BOOST minority importance
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

# ======================
# TRAIN
# ======================
lr.fit(X_train_scaled, y_train)
rf.fit(X_train, y_train)
xgb.fit(X_train, y_train)

# ======================
# PREDICTIONS
# ======================
lr_pred = lr.predict(X_test_scaled)
rf_pred = rf.predict(X_test)
xgb_pred = xgb.predict(X_test)
print("Unique predictions:", np.unique(xgb_pred))

# ======================
# THRESHOLD TUNING (IMPORTANT)
# ======================
y_proba = xgb.predict_proba(X_test)[:, 1]

threshold = 0.3  # you can tune this
y_pred_custom = (y_proba > threshold).astype(int)

print("\n===== XGBoost (Custom Threshold) =====")
print(classification_report(y_test, y_pred_custom))

# ======================
# EVALUATION FUNCTION
# ======================
def evaluate(name, y_true, y_pred):
    print(f"\n===== {name} =====")
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("ROC-AUC:", roc_auc_score(y_true, y_pred))
    print(classification_report(y_true, y_pred))

# ======================
# EVALUATE
# ======================
evaluate("Logistic Regression", y_test, lr_pred)
evaluate("Random Forest", y_test, rf_pred)
evaluate("XGBoost", y_test, xgb_pred)

# ======================
# FEATURE IMPORTANCE
# ======================
importances = pd.Series(rf.feature_importances_, index=X.columns)
print("\nTop Features:\n", importances.sort_values(ascending=False).head(10))

# ======================
# FINAL CHECK
# ======================
print("\nAny NaN:", X_train.isnull().sum().sum())
print("Any inf:", np.isinf(X_train).sum().sum())

# ======================
# AFTER MODEL TRAINING & EVALUATION
# ======================
import joblib
import os

os.makedirs("model", exist_ok=True)

joblib.dump(xgb, "model/xgb_model.pkl")
joblib.dump(list(X.columns), "model/features.pkl")

print("✅ NEW MODEL SAVED")

e:\bankruptcy-risk-system\venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [00:07:18] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Unique predictions: [0 1]

===== XGBoost (Custom Threshold) =====
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1001
           1       1.00      0.99      0.99        70

    accuracy                           1.00      1071
   macro avg       1.00      0.99      1.00      1071
weighted avg       1.00      1.00      1.00      1071


===== Logistic Regression =====
Accuracy: 0.8683473389355743
ROC-AUC: 0.8365634365634366
              precision    recall  f1-score   support

           0       0.98      0.87      0.93      1001
           1       0.31      0.80      0.44        70

    accuracy                           0.87      1071
   macro avg       0.65      0.84      0.68      1071
weighted avg       0.94      0.87      0.89      1071


===== Random Forest =====
Accuracy: 0.9990662931839402
ROC-AUC: 0.9928571428571429
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      10

In [20]:
#new
import pandas as pd
import numpy as np
import os

# ======================
# LOAD DATA
# ======================
df = pd.read_csv(r"E:\bankruptcy-risk-system\data\processed\cleaned_data.csv")

# ======================
# CLEAN COLUMN NAMES
# ======================
df.columns = df.columns.str.encode('ascii', 'ignore').str.decode('ascii')
df.columns = df.columns.str.strip().str.replace(" ", "_").str.replace(".", "").str.replace("/", "_")

# ======================
# RENAME COLUMNS
# ======================
df.rename(columns={
    "DebtRsCr": "debt",
    "CF_OperationsRsCr": "cash_flow",
    "ROA_12M%": "roa",
    "ROCE%": "roce"
}, inplace=True)

# ======================
# HANDLE MISSING VALUES
# ======================
df = df.replace([np.inf, -np.inf], np.nan)

num_cols = df.select_dtypes(include=np.number).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# ======================
# 🔴 FINANCIAL FEATURES
# ======================
df["cash_flow_stress"] = df["debt"] / (abs(df["cash_flow"]) + 1)

df["liquidity_risk"] = 1 / (df["current_ratio"] + 1)

df["fsi"] = (
    df["current_ratio"] +
    df["roa"] +
    df["roce"] -
    df["debt_to_equity"]
)

# ======================
# 🔵 BUSINESS FEATURES
# ======================
df["cost_pressure"] = 1 / (df["roce"] + 1)

df["operational_inefficiency"] = 1 / (df["roce"] + 1)

df["competitive_pressure"] = df["cost_pressure"] + df["operational_inefficiency"]

df["profitability"] = df["roa"] + df["roce"]

# ======================
# 🟣 MANAGEMENT FEATURES
# ======================
df["management_risk"] = (
    (df["roa"] < df["roa"].median()) &
    (df["cash_flow"] < df["cash_flow"].median())
).astype(int)

# ======================
# 🎯 TARGET (REALISTIC + NO LEAKAGE)
# ======================
df["default_flag"] = (
    (df["cash_flow_stress"] > df["cash_flow_stress"].quantile(0.7)) &
    (df["liquidity_risk"] > df["liquidity_risk"].quantile(0.7)) &
    (df["profitability"] < df["profitability"].quantile(0.3))
).astype(int)

# ======================
# 🔥 ADD NOISE (VERY IMPORTANT)
# ======================
noise_idx = df.sample(frac=0.1, random_state=42).index
df.loc[noise_idx, "default_flag"] = 1 - df.loc[noise_idx, "default_flag"]

# ======================
# CHECK TARGET
# ======================
print("\nTarget Distribution:")
print(df["default_flag"].value_counts())

# ======================
# FINAL FEATURE SET
# ======================
final_df = df[[
    "company_name",
    "cash_flow_stress",
    "liquidity_risk",
    "fsi",
    "cost_pressure",
    "operational_inefficiency",
    "competitive_pressure",
    "profitability",
    "management_risk",
    "default_flag"
]]

# ======================
# SAVE
# ======================
BASE_DIR = r"E:\bankruptcy-risk-system"
os.makedirs(os.path.join(BASE_DIR, "data/processed"), exist_ok=True)

final_df.to_csv(os.path.join(BASE_DIR, "data/processed/final_features.csv"), index=False)

print("\n✅ final_features.csv created successfully")

# ======================
# SANITY CHECKS
# ======================
print("\nAny nulls:\n", final_df.isnull().sum())
print("\nAny infinite values:\n", np.isinf(final_df.select_dtypes(include=np.number)).sum())


Target Distribution:
default_flag
0    4550
1     803
Name: count, dtype: int64

✅ final_features.csv created successfully

Any nulls:
 company_name                0
cash_flow_stress            0
liquidity_risk              0
fsi                         0
cost_pressure               0
operational_inefficiency    0
competitive_pressure        0
profitability               0
management_risk             0
default_flag                0
dtype: int64

Any infinite values:
 cash_flow_stress            0
liquidity_risk              0
fsi                         0
cost_pressure               1
operational_inefficiency    1
competitive_pressure        1
profitability               0
management_risk             0
default_flag                0
dtype: int64


In [21]:
#new model
import pandas as pd
import numpy as np
import os
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# ======================
# LOAD DATA
# ======================
df = pd.read_csv(r"E:\bankruptcy-risk-system\data\processed\final_features.csv")

# ======================
# SPLIT FEATURES & TARGET
# ======================
X = df.drop(columns=["company_name", "default_flag"])
y = df["default_flag"]

# ======================
# TRAIN TEST SPLIT
# ======================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ======================
# PREPROCESSING (NO LEAKAGE)
# ======================

# Train
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_train = X_train.clip(lower=-1e6, upper=1e6)

train_median = X_train.median()
X_train = X_train.fillna(train_median)

# Test (use SAME median)
X_test = X_test.replace([np.inf, -np.inf], np.nan)
X_test = X_test.clip(lower=-1e6, upper=1e6)
X_test = X_test.fillna(train_median)

# ======================
# SCALING (for LR only)
# ======================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ======================
# MODELS
# ======================

# Logistic Regression
lr = LogisticRegression(class_weight='balanced', max_iter=1000)

# Random Forest
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    class_weight='balanced',
    random_state=42
)

# XGBoost (FINAL FIXED VERSION)
scale_pos_weight = len(y_train[y_train == 0]) / len(y_train[y_train == 1])

xgb = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=scale_pos_weight * 3,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    gamma=1,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

# ======================
# TRAIN MODELS
# ======================
lr.fit(X_train_scaled, y_train)
rf.fit(X_train, y_train)
xgb.fit(X_train, y_train)

# ======================
# PREDICTIONS
# ======================
lr_pred = lr.predict(X_test_scaled)
rf_pred = rf.predict(X_test)
xgb_pred = xgb.predict(X_test)

# 🔥 IMPORTANT CHECK
print("Unique predictions (XGB):", np.unique(xgb_pred))

# ======================
# THRESHOLD TUNING
# ======================
y_proba = xgb.predict_proba(X_test)[:, 1]

threshold = 0.3
y_pred_custom = (y_proba > threshold).astype(int)

print("\n===== XGBoost (Custom Threshold) =====")
print(classification_report(y_test, y_pred_custom))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

# ======================
# EVALUATION FUNCTION
# ======================
def evaluate(name, y_true, y_pred):
    print(f"\n===== {name} =====")
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("ROC-AUC:", roc_auc_score(y_true, y_pred))
    print(classification_report(y_true, y_pred))

# ======================
# EVALUATE ALL MODELS
# ======================
evaluate("Logistic Regression", y_test, lr_pred)
evaluate("Random Forest", y_test, rf_pred)
evaluate("XGBoost", y_test, xgb_pred)

# ======================
# FEATURE IMPORTANCE
# ======================
importances = pd.Series(rf.feature_importances_, index=X.columns)
print("\nTop Features:\n", importances.sort_values(ascending=False).head(10))

# ======================
# FINAL CHECKS
# ======================
print("\nAny NaN:", X_train.isnull().sum().sum())
print("Any inf:", np.isinf(X_train).sum().sum())

# ======================
# SAVE MODEL
# ======================
os.makedirs("model", exist_ok=True)

joblib.dump(xgb, "model/xgb_model.pkl")
joblib.dump(list(X.columns), "model/features.pkl")

print("\n✅ MODEL SAVED SUCCESSFULLY")

e:\bankruptcy-risk-system\venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [00:13:43] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Unique predictions (XGB): [0 1]

===== XGBoost (Custom Threshold) =====
              precision    recall  f1-score   support

           0       0.89      0.57      0.69       910
           1       0.20      0.60      0.30       161

    accuracy                           0.57      1071
   macro avg       0.54      0.58      0.50      1071
weighted avg       0.79      0.57      0.63      1071

ROC-AUC: 0.6721179441676336

===== Logistic Regression =====
Accuracy: 0.7002801120448179
ROC-AUC: 0.634472049689441
              precision    recall  f1-score   support

           0       0.90      0.73      0.81       910
           1       0.26      0.54      0.35       161

    accuracy                           0.70      1071
   macro avg       0.58      0.63      0.58      1071
weighted avg       0.80      0.70      0.74      1071


===== Random Forest =====
Accuracy: 0.8795518207282913
ROC-AUC: 0.6453893932154801
              precision    recall  f1-score   support

           0      